## BERT 모델 학습 및 결과 테스트

In [ ]:
!pip -q install transformers datasets pyarrow

In [2]:
import torch
import numpy as np
from datasets import load_dataset
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

transformers.logging.set_verbosity_error()

토크나이저 및 모델 로드

In [3]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def predict_sentiment(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        
    predicted_class = torch.argmax(outputs.logits, dim=-1).item()
    return "긍정 (Positive)" if predicted_class == 1 else "부정 (Negative)"


학습 전 모델 테스트

In [4]:
test_sentence = "I really wanted to like this movie, but it fell completely flat."

print("학습 전 모델 추론")
print(f"입력 문장: {test_sentence}")
print(f"예측 결과: {predict_sentiment(test_sentence)}") 


학습 전 모델 추론
입력 문장: I really wanted to like this movie, but it fell completely flat.
예측 결과: 긍정 (Positive)


데이터셋 로드

In [5]:
print("데이터셋 준비 중...")
dataset = load_dataset("imdb")
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(1000))
small_test_dataset = dataset["test"].shuffle(seed=42).select(range(200))

small_train_dataset

데이터셋 준비 중...


Dataset({
    features: ['text', 'label'],
    num_rows: 1000
})

In [6]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = small_train_dataset.map(tokenize_function, batched=True, desc="Train Tokenization")
tokenized_test = small_test_dataset.map(tokenize_function, batched=True, desc="Test Tokenization")

학습

In [7]:
print("\n모델 학습 시작...")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": (predictions == labels).astype(np.float32).mean().item()}

training_args = TrainingArguments(
    output_dir="test_trainer",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    report_to="none" # 로그 깔끔하게 정리
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()
print("\n모델 학습 완료.\n")



모델 학습 시작...
{'eval_loss': 0.46590477228164673, 'eval_accuracy': 0.7850000262260437, 'eval_runtime': 0.3642, 'eval_samples_per_second': 549.217, 'eval_steps_per_second': 35.699, 'epoch': 1.0}
{'eval_loss': 0.40550491213798523, 'eval_accuracy': 0.8100000023841858, 'eval_runtime': 0.3913, 'eval_samples_per_second': 511.126, 'eval_steps_per_second': 33.223, 'epoch': 2.0}
{'eval_loss': 0.42244279384613037, 'eval_accuracy': 0.7950000166893005, 'eval_runtime': 0.3462, 'eval_samples_per_second': 577.726, 'eval_steps_per_second': 37.552, 'epoch': 3.0}
{'train_runtime': 21.7449, 'train_samples_per_second': 137.963, 'train_steps_per_second': 8.692, 'train_loss': 0.4083800542922247, 'epoch': 3.0}

모델 학습 완료.



모델 추론

In [8]:
print("학습 후 모델 추론")
print(f"입력 문장: {test_sentence}")
print(f"예측 결과: {predict_sentiment(test_sentence)}")

학습 후 모델 추론
입력 문장: I really wanted to like this movie, but it fell completely flat.
예측 결과: 부정 (Negative)
